In [1]:
import torch
from torchvision import datasets, transforms
from torchvision.models import (
    densenet121, resnet50, convnext_small, swin_t, vit_b_16,
    efficientnet_v2_s, mobilenet_v3_large
)
from PIL import Image
from torch.utils.data import DataLoader
import os
from tqdm import tqdm

In [2]:
VAL_DIR = r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val"
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
models_config = {
    "ResNet50": {
        "model": resnet50,
        "weights": "../weights/ResNet50.pth"
    },
    "ConvNeXt": {
        "model": convnext_small,
        "weights": "../weights/ConvNeXt-S.pth"
    },
    "SwinT": {
        "model": swin_t,
        "weights": "../weights/Swin-T.pth"
    },
    "EfficientNetV2": {
        "model": efficientnet_v2_s,
        "weights": "../weights/EfficientNet-v2.pth"
    },
    "Vit-B": {
        "model": vit_b_16,
        "weights": "../weights/Vit-B.pth"
    },
    "MobileNetV3": {
        "model": mobilenet_v3_large,
        "weights": "../weights/MobileNet-v3.pth"
    },
    "DenseNet121": {
        "model": densenet121,
        "weights": "../weights/DenseNet121.pth"
    }
}

loaded_models = {}
class_names = None

for name, cfg in models_config.items():
    checkpoint = torch.load(cfg["weights"], map_location=device)
    
    # Extract class_names from first checkpoint (same for all)
    if class_names is None and "class_names" in checkpoint:
        class_names = checkpoint["class_names"]
    
    # Create model
    model = cfg["model"](weights=None)
    num_classes = len(class_names) if class_names else checkpoint["model_state"].get("classifier.1.weight", torch.tensor([0])).shape[0]
    
    # Adjust classifier head
    if name == "ResNet50":
        model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
    elif name == "ConvNeXt":
        model.classifier[2] = torch.nn.Linear(model.classifier[2].in_features, num_classes)
    elif name == "SwinT":
        model.head = torch.nn.Linear(model.head.in_features, num_classes)
    elif name == "Vit-B":
        model.heads.head = torch.nn.Linear(model.heads.head.in_features, num_classes)
    elif name == "EfficientNetV2":
        model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, num_classes)
    elif name == "MobileNetV3":
        model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, num_classes)
    elif name == "DenseNet121":
        model.classifier = torch.nn.Linear(model.classifier.in_features, num_classes)
    
    # Load state dict
    state_key = "model_state_dict" if "model_state_dict" in checkpoint else "model_state"
    model.load_state_dict(checkpoint[state_key], strict=True)
    model.to(device)
    model.eval()
    
    loaded_models[name] = model

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [5]:
val_dataset = datasets.ImageFolder(VAL_DIR, transform=transform)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = val_dataset.classes

In [6]:
val_accuracies = {
    "ResNet50": 0.9750,
    "ConvNeXt": 0.9750,
    "SwinT": 0.9583,
    "Vit-B": 0.9625,
    "EfficientNetV2": 0.8750,
    "MobileNetV3": 0.9375,
    "DenseNet121": 0.9750
}

total_acc = sum(val_accuracies.values())
weights = {
    name: acc / total_acc
    for name, acc in val_accuracies.items()
}


In [7]:
@torch.no_grad()
def evaluate_ensemble(val_loader, loaded_models):
    correct = 0
    total = 0

    for images, labels in tqdm(val_loader, desc="Hard Voting Validating "):
        images = images.to(device)
        labels = labels.to(device)

        batch_size = images.size(0)
        num_classes = len(val_loader.dataset.classes)

        # Tensor to store vote counts per class
        votes = torch.zeros(batch_size, num_classes).to(device)

        for name, model in loaded_models.items():
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            # Add 1 vote for predicted class
            votes[torch.arange(batch_size), preds] += 1

        # Final prediction = class with most votes
        final_preds = torch.argmax(votes, dim=1)

        correct += (final_preds == labels).sum().item()
        total += labels.size(0)

    return correct / total

In [8]:
ensemble_acc = evaluate_ensemble(val_loader, loaded_models)

print("====================================")
print(f"Ensemble Validation Accuracy: {ensemble_acc:.4f}")
print("====================================")

Hard Voting Validating : 100%|██████████| 8/8 [02:10<00:00, 16.25s/it]

Ensemble Validation Accuracy: 0.9667
